<a href="https://colab.research.google.com/github/silvanamolano17-del/cinema-management-cpp./blob/main/UniSmart_%E2%80%94_ACL_entre_Perfil_y_Predicci%C3%B3n_Acad%C3%A9mica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# ============================================================
# ANTI-CORRUPTION LAYER (ACL) — Subdominio: Predicción
# ============================================================
# Patrón DDD: Anti-Corruption Layer
# Propósito: traducir el modelo de datos de Perfil (Upstream)
# al modelo interno de Predicción (Downstream).
# Si Perfil cambia su estructura, solo se modifica esta clase,
# el modelo ML nunca se ve afectado.
# ============================================================

class PerfilTraductor:

    def traducir(self, perfil_raw: dict) -> dict:
        """
        Recibe el JSON tal como lo devuelve la API de Perfil
        y lo convierte al formato que espera PredictorDeRendimiento.

        Parámetros:
            perfil_raw: datos crudos del subdominio Perfil (Upstream)

        Retorna:
            dict con 'student_features' — modelo interno de Predicción
        """
        # Extrae la lista de materias cursadas, vacía si no existe
        materias = perfil_raw.get("materias_cursadas", [])

        # Construye el modelo interno con los features que necesita el ML
        return {
            "student_features": {
                "avg_grade":     self._promedio(materias),  # promedio histórico de notas
                "courses_taken": len(materias),             # total de materias cursadas
                "current_load":  perfil_raw.get("carga_actual", 0),   # materias este semestre
                "sleep_hours":   perfil_raw.get("horas_sueno", 7),    # horas de sueño reportadas
                "semester":      perfil_raw.get("semestre_actual", 1), # semestre en curso
            }
        }

    def _promedio(self, materias: list) -> float:
        """
        Calcula el promedio de notas del historial académico.
        Método privado — solo lo usa esta clase.

        Parámetros:
            materias: lista de materias cursadas con sus notas

        Retorna:
            promedio redondeado a 2 decimales, 0.0 si no hay materias
        """
        if not materias:
            return 0.0  # evita división por cero si el historial está vacío

        notas = [m.get("nota_final", 0) for m in materias]  # extrae solo las notas
        return round(sum(notas) / len(notas), 2)

In [9]:
# ============================================================
# NÚCLEO DEL DOMINIO — Subdominio: Predicción Académica
# ============================================================
# Patrón DDD: Hexagonal Architecture (núcleo aislado)
# Propósito: calcular el ScoreDeAprobación de un estudiante.
# Esta clase NO importa nada del subdominio Perfil.
# Solo trabaja con 'student_features' — su modelo interno.
# En producción, self.modelo sería un Random Forest o XGBoost
# entrenado con datos reales de estudiantes de la universidad.
# ============================================================

class PredictorDeRendimiento:

    def predecir(self, student_features: dict) -> float:
        """
        Calcula la probabilidad de que el estudiante apruebe
        una materia con nota mayor a 3.5.

        Parámetros:
            student_features: modelo interno ya traducido por la ACL

        Retorna:
            ScoreDeAprobación: float entre 0.0 y 1.0
            (0 = muy probable que repruebe, 1 = muy probable que apruebe)
        """
        # Extrae los valores de los features en orden
        features = list(student_features["student_features"].values())

        # SIMULACIÓN: usa el promedio normalizado sobre 5.0
        # En producción esta línea sería: self.modelo.predict([features])[0]
        avg_grade = features[0]
        score = min(avg_grade / 5.0, 1.0)

        return round(score, 2)  # redondea a 2 decimales)

In [10]:
# ============================================================
# PRUEBA DEL FLUJO COMPLETO — ACL + Predictor
# ============================================================
# Simula el flujo real del sistema:
# 1. Perfil devuelve datos crudos (como lo haría su API)
# 2. La ACL traduce esos datos al modelo interno
# 3. El Predictor calcula el ScoreDeAprobación
# 4. Si el score < 0.4 se dispara RiesgoAcadémicoDetectado
# ============================================================

# --- Paso 1: datos simulados como si vinieran de la API de Perfil ---
perfil_raw = {
    "estudiante_id": "12345",
    "materias_cursadas": [
        {"codigo": "ISIS1101", "nota_final": 4.2, "periodo": "2024-1"},
        {"codigo": "MATE1100", "nota_final": 3.8, "periodo": "2024-1"},
        {"codigo": "ISIS1300", "nota_final": 4.5, "periodo": "2024-2"},
    ],
    "carga_actual": 4,    # materias inscritas este semestre
    "horas_sueno": 6,     # horas de sueño promedio reportadas
    "semestre_actual": 3  # semestre que cursa actualmente
}

# --- Paso 2: instanciar las clases del dominio ---
traductor = PerfilTraductor()       # ACL: traduce de Perfil → Predicción
predictor = PredictorDeRendimiento() # Núcleo: calcula el score

# --- Paso 3: traducir los datos crudos usando la ACL ---
# Después de este paso, el predictor ya no sabe nada de cómo
# Perfil estructura sus datos — solo ve student_features
perfil_traducido = traductor.traducir(perfil_raw)

# --- Paso 4: predecir el rendimiento ---
score = predictor.predecir(perfil_traducido)

# --- Paso 5: mostrar resultados ---
print("=== Datos crudos de Perfil (Upstream) ===")
print(perfil_raw)

print("\n=== Perfil traducido por ACL (modelo interno) ===")
print(perfil_traducido)

print(f"\n=== Resultado de Predicción ===")
print(f"ScoreDeAprobación: {score}")

# Umbral de riesgo académico definido en el dominio
UMBRAL_RIESGO = 0.4

if score < UMBRAL_RIESGO:
    # En el sistema real aquí se publicaría el evento
    # RiesgoAcadémicoDetectado en RabbitMQ
    print(f"⚠️  EVENTO: RiesgoAcadémicoDetectado — score {score} < umbral {UMBRAL_RIESGO}")
else:
    print(f"✅  Estudiante en buen rendimiento académico")

=== Datos crudos de Perfil (Upstream) ===
{'estudiante_id': '12345', 'materias_cursadas': [{'codigo': 'ISIS1101', 'nota_final': 4.2, 'periodo': '2024-1'}, {'codigo': 'MATE1100', 'nota_final': 3.8, 'periodo': '2024-1'}, {'codigo': 'ISIS1300', 'nota_final': 4.5, 'periodo': '2024-2'}], 'carga_actual': 4, 'horas_sueno': 6, 'semestre_actual': 3}

=== Perfil traducido por ACL (modelo interno) ===
{'student_features': {'avg_grade': 4.17, 'courses_taken': 3, 'current_load': 4, 'sleep_hours': 6, 'semester': 3}}

=== Resultado de Predicción ===
ScoreDeAprobación: 0.83
✅  Estudiante en buen rendimiento académico
